# 9th week's homework - by [Aleksei Novikov](https://www.linkedin.com/in/devnovikov/)

Serverless Deep Learning - Deploying Hair Classifier

## Preparation

In [16]:
import onnxruntime as ort
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image

## Preparing the Image

Helper functions for downloading and resizing images.

In [17]:
def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img

## Question 1: Output Node Name

To be able to use this model, we need to know the name of the input and output nodes.

What's the name of the output?

In [18]:
session = ort.InferenceSession('hair_classifier_v1.onnx')

print("Input details:")
for input_info in session.get_inputs():
    print(f"  Name: {input_info.name}")
    print(f"  Shape: {input_info.shape}")
    print(f"  Type: {input_info.type}")

print("Output details:")
for output_info in session.get_outputs():
    print(f"  Name: {output_info.name}")
    print(f"  Shape: {output_info.shape}")
    print(f"  Type: {output_info.type}")

Input details:
  Name: input
  Shape: ['s77', 3, 200, 200]
  Type: tensor(float)
Output details:
  Name: output
  Shape: ['s77', 1]
  Type: tensor(float)


**Answer: `output`**

## Question 2: Target Size

Based on the previous homework, what should be the target size for the image?

In [20]:
url = "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"
img = download_image(url)

print(f"Original image size: {img.size}")

target_size = (200, 200)
img = prepare_image(img, target_size)
print(f"Resized image size: {img.size}")

Original image size: (1024, 1024)
Resized image size: (200, 200)


From homework 8, we used 200x200 images for training the hair classifier.
The model input shape also confirms: `['s77', 3, 200, 200]`

**Answer: 200x200**

## Question 3: Preprocessing

Now we need to turn the image into numpy array and pre-process it.

From homework 8, we used ImageNet normalization:
- mean = [0.485, 0.456, 0.406]
- std = [0.229, 0.224, 0.225]

After preprocessing, what's the value in the first pixel, the R channel?

In [27]:
x = np.array(img, dtype=np.float32) / 255.0

print(f"Array shape after conversion: {x.shape}")
print(f"Value range: [{x.min():.4f}, {x.max():.4f}]")

# Apply ImageNet normalization
mean = np.array([0.485, 0.456, 0.406])
std = np.array([0.229, 0.224, 0.225])

x = (x - mean) / std

print("\n")
print(f"After normalization:")
print(f"Value range: [{x.min():.4f}, {x.max():.4f}]")

# First pixel, R channel value
first_pixel_r = x[0, 0, 0]
print(f"First pixel R channel value: {first_pixel_r:.4f}")

Array shape after conversion: (200, 200, 3)
Value range: [0.0000, 1.0000]


After normalization:
Value range: [-2.1179, 2.6400]
First pixel R channel value: -1.0733


**Answer: -1.073**

## Question 4: Model Output

Now let's apply this model to this image. What's the output of the model?

In [30]:
# Prepare input
# Current shape is (H, W, C), need (N, C, H, W)
x_input = np.transpose(x, (2, 0, 1))  # (C, H, W)
x_input = np.expand_dims(x_input, axis=0)  # (N, C, H, W)

print(f"Input shape for model: {x_input.shape}")

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"Input name: {input_name}")
print(f"Output name: {output_name}")

output = session.run([output_name], {input_name: x_input.astype(np.float32)})[0]

print(f"Model output: {output}")
print(f"Output value: {output[0][0]:.4f}")

Input shape for model: (1, 3, 200, 200)
Input name: input
Output name: output
Model output: [[0.08927515]]
Output value: 0.0893


**Answer: 0.09**

## Docker Questions

For Questions 5 and 6, we'll work with Docker.

## Question 5: Docker Image Size

Download the base image `agrigorev/model-2025-hairstyle:v1` and check its size.

```bash
docker pull agrigorev/model-2025-hairstyle:v1
docker images agrigorev/model-2025-hairstyle:v1
```

In [32]:
# docker pull agrigorev/model-2025-hairstyle:v1
# docker images agrigorev/model-2025-hairstyle:v1

# Output:
# REPOSITORY                       TAG       IMAGE ID       CREATED      SIZE
# agrigorev/model-2025-hairstyle   v1        4528ad1525d5   6 days ago   608MB

print("REPOSITORY                       TAG       IMAGE ID       CREATED      SIZE")
print("agrigorev/model-2025-hairstyle   v1        4528ad1525d5   6 days ago   608MB")

REPOSITORY                       TAG       IMAGE ID       CREATED      SIZE
agrigorev/model-2025-hairstyle   v1        4528ad1525d5   6 days ago   608MB


**Answer: 608 Mb**

## Question 6: Lambda Container Output

Extend the docker image, install required libraries, add lambda code, and run the container locally.

### Dockerfile
```dockerfile
FROM agrigorev/model-2025-hairstyle:v1

RUN pip install pillow onnxruntime

COPY lambda_function.py .

CMD ["lambda_function.lambda_handler"]
```

### lambda_function.py
```python
import onnxruntime as ort
import numpy as np
from io import BytesIO
from urllib import request
from PIL import Image

session = ort.InferenceSession('hair_classifier_empty.onnx')
input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name


def download_image(url):
    with request.urlopen(url) as resp:
        buffer = resp.read()
    stream = BytesIO(buffer)
    img = Image.open(stream)
    return img


def prepare_image(img, target_size):
    if img.mode != 'RGB':
        img = img.convert('RGB')
    img = img.resize(target_size, Image.NEAREST)
    return img


def preprocess(img):
    x = np.array(img, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    x = (x - mean) / std
    x = np.transpose(x, (2, 0, 1))
    x = np.expand_dims(x, axis=0).astype(np.float32)
    return x


def predict(url):
    img = download_image(url)
    img = prepare_image(img, target_size=(200, 200))
    x = preprocess(img)
    output = session.run([output_name], {input_name: x})[0]
    return float(output[0][0])


def lambda_handler(event, context):
    url = event['url']
    result = predict(url)
    return {'prediction': result}
```

### Build and Run
```bash
docker build -t hair-lambda -f Dockerfile .
docker run -p 9000:8080 hair-lambda
```

### Test
```bash
curl -X POST "http://localhost:9000/2015-03-31/functions/function/invocations" \
  -d '{"url": "https://habrastorage.org/webt/yf/_d/ok/yf_dokzqy3vcritme8ggnzqlvwa.jpeg"}'
```

In [34]:
# Test result:
# {"prediction": -0.1022084578871727}

print('Response: {"prediction": -0.1022084578871727}')

Response: {"prediction": -0.1022084578871727}


**Answer: -0.10**

## Summary of Answers

| Question | Answer |
|----------|--------|
| Q1: Output node name | `output` |
| Q2: Target size | 200x200 |
| Q3: First pixel R value | -1.073 |
| Q4: Model output | 0.09 |
| Q5: Docker image size | 608 Mb |
| Q6: Lambda output | -0.10 |